# Financial Data Quality Audit — SEC EDGAR Multi-Company Review

**Author:** Alejandro (Alex)  
**GitHub:** chooseyourtacoday  
**Data Source:** SEC EDGAR XBRL API (`https://data.sec.gov`)  
**Companies:** Apple (AAPL), Microsoft (MSFT), Tesla (TSLA), Amazon (AMZN), Johnson & Johnson (JNJ)  
**Output:** Excel audit report with issue log, severity ratings, and per-company data quality scorecards

---

### What This Audit Does

Financial data from SEC EDGAR is authoritative — but it is not clean by default. Companies refile quarters, use inconsistent XBRL tags across years, report cumulative YTD figures alongside quarterly figures, and occasionally submit values in different units. This audit systematically identifies every category of data quality issue before any downstream analysis begins.

**Seven audit check categories:**

| # | Category | What It Checks |
|---|---|---|
| 1 | **Completeness** | Missing quarters, null fields, expected vs. actual row counts |
| 2 | **Mathematical Consistency** | Revenue − COGS = Gross Profit? Margins recalculate correctly? |
| 3 | **Cross-Filing Reconciliation** | Q1+Q2+Q3+Q4 sums match annual 10-K reported figures? |
| 4 | **Temporal Continuity** | Gaps between filing dates, out-of-sequence periods |
| 5 | **Anomaly Detection** | Values outside 3σ, QoQ swings >50%, sign reversals |
| 6 | **XBRL Tag Consistency** | Same concept reported under different tags across years |
| 7 | **Duplicate Detection** | Same period filed more than once |

Each issue is logged with: company, metric, period, check category, severity (Critical / High / Medium / Low), expected value, actual value, and recommended remediation.

---

### Why This Matters

Most junior analysts skip straight to analysis. A single undetected data quality issue — a refiled quarter that doubled a revenue figure, a cumulative YTD value mistaken for a quarterly one — corrupts every downstream calculation silently. This audit runs first, every time, before any model touches the data.

This is the workflow used in professional financial data roles: **audit → document → remediate → analyze.**

---
## Section 1: Setup

In [1]:
import requests
import pandas as pd
import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

print(f"Audit initialized: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Audit initialized: 2026-05-29 08:43


In [2]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
COMPANIES = {
    'Apple':             {'cik': '0000320193', 'ticker': 'AAPL', 'fiscal_year_end': 'September'},
    'Microsoft':         {'cik': '0000789019', 'ticker': 'MSFT', 'fiscal_year_end': 'June'},
    'Tesla':             {'cik': '0001318605', 'ticker': 'TSLA', 'fiscal_year_end': 'December'},
    'Amazon':            {'cik': '0001018724', 'ticker': 'AMZN', 'fiscal_year_end': 'December'},
    'Johnson & Johnson': {'cik': '0000200406', 'ticker': 'JNJ',  'fiscal_year_end': 'December'},
}

# Metrics to audit — primary tag + known fallback tags per company
METRICS = {
    'Revenue':          ['Revenues', 'SalesRevenueNet', 'RevenueFromContractWithCustomerExcludingAssessedTax'],
    'Gross Profit':     ['GrossProfit'],
    'Operating Income': ['OperatingIncomeLoss'],
    'Net Income':       ['NetIncomeLoss'],
    'EPS Diluted':      ['EarningsPerShareDiluted'],
}

# Severity thresholds
THRESHOLDS = {
    'qoq_swing_pct':     0.50,   # >50% QoQ change triggers anomaly check
    'math_tolerance':    0.02,   # 2% tolerance on consistency checks (rounding)
    'sigma_threshold':   3.0,    # Values beyond 3σ flagged as anomalies
    'recon_tolerance':   0.05,   # 5% tolerance on quarterly-to-annual reconciliation
}

HEADERS = {'User-Agent': 'Portfolio Audit Project analyst@example.com'}
AUDIT_DATE = datetime.now().strftime('%Y-%m-%d')

print(f"Companies: {len(COMPANIES)}  |  Metrics: {len(METRICS)}  |  Check categories: 7")

Companies: 5  |  Metrics: 5  |  Check categories: 7


---
## Section 2: Issue Log Framework

Every check in this audit writes findings to a central issue log. Each issue captures:
- **Who** — company and ticker
- **What** — metric and period affected
- **Which check** — the audit category that caught it
- **Severity** — Critical, High, Medium, or Low
- **Evidence** — expected vs. actual values
- **Remediation** — recommended corrective action

In [3]:
# Central issue log — all checks write here
ISSUES = []

def log_issue(company, ticker, metric, period, check_category,
               severity, description, expected=None, actual=None, remediation=None):
    """
    Append a data quality finding to the central issue log.
    
    Severity levels:
        Critical — data is unusable as-is; will corrupt downstream analysis
        High     — significant anomaly requiring investigation before use
        Medium   — notable irregularity; use with caution
        Low      — minor inconsistency; document and monitor
    """
    ISSUES.append({
        'Audit Date':      AUDIT_DATE,
        'Company':         company,
        'Ticker':          ticker,
        'Metric':          metric,
        'Period':          str(period),
        'Check Category':  check_category,
        'Severity':        severity,
        'Description':     description,
        'Expected Value':  str(expected) if expected is not None else 'N/A',
        'Actual Value':    str(actual)   if actual   is not None else 'N/A',
        'Remediation':     remediation or 'Review source filing on SEC EDGAR',
    })

# Severity weights for scoring
SEVERITY_WEIGHTS = {'Critical': 10, 'High': 5, 'Medium': 2, 'Low': 1}

print("Issue log framework initialized.")

Issue log framework initialized.


---
## Section 3: Data Ingestion

Pull raw data from SEC EDGAR for all companies and metrics. Minimal transformation — we want the raw state so the audit sees what any downstream analyst would receive.

In [4]:
def fetch_concept_raw(cik, concept):
    """
    Pull raw XBRL concept data from EDGAR — NO filtering, NO deduplication.
    Returns the full unprocessed response for audit purposes.
    """
    url = f'https://data.sec.gov/api/xbrl/companyconcept/CIK{cik}/us-gaap/{concept}.json'
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        if r.status_code != 200:
            return None
        data = r.json()
        units = data.get('units', {})
        usd = units.get('USD') or units.get('USD/shares')
        if not usd:
            return None
        df = pd.DataFrame(usd)
        df['end'] = pd.to_datetime(df['end'])
        if 'start' in df.columns:
            df['start'] = pd.to_datetime(df['start'])
        df['concept_tag'] = concept
        return df
    except Exception as e:
        print(f"  Error: {e}")
        return None


def ingest_company(company_name, info):
    """Pull all metrics for one company. Returns dict of raw DataFrames by metric."""
    print(f"\n{company_name} ({info['ticker']})")
    cik = info['cik']
    raw = {}

    for metric, tags in METRICS.items():
        for tag in tags:
            df = fetch_concept_raw(cik, tag)
            time.sleep(0.15)
            if df is not None and len(df) > 0:
                raw[metric] = df
                print(f"  ✓ {metric} via '{tag}' ({len(df)} raw rows)")
                break
        else:
            print(f"  ✗ {metric} — no data found")
            log_issue(company_name, info['ticker'], metric, 'ALL',
                      'Completeness', 'Critical',
                      f'No data returned for {metric} across all known concept tags',
                      remediation='Check SEC EDGAR for correct XBRL tag; company may use non-standard taxonomy')
    return raw


# Pull all data
ALL_RAW = {}
for co_name, co_info in COMPANIES.items():
    ALL_RAW[co_name] = ingest_company(co_name, co_info)

print(f"\n{'='*50}")
print("Ingestion complete. Beginning audit checks.")


Apple (AAPL)
  ✓ Revenue via 'Revenues' (11 raw rows)
  ✓ Gross Profit via 'GrossProfit' (334 raw rows)
  ✓ Operating Income via 'OperatingIncomeLoss' (230 raw rows)
  ✓ Net Income via 'NetIncomeLoss' (334 raw rows)
  ✓ EPS Diluted via 'EarningsPerShareDiluted' (334 raw rows)

Microsoft (MSFT)
  ✓ Revenue via 'Revenues' (31 raw rows)
  ✓ Gross Profit via 'GrossProfit' (303 raw rows)
  ✓ Operating Income via 'OperatingIncomeLoss' (275 raw rows)
  ✓ Net Income via 'NetIncomeLoss' (337 raw rows)
  ✓ EPS Diluted via 'EarningsPerShareDiluted' (337 raw rows)

Tesla (TSLA)
  ✓ Revenue via 'Revenues' (278 raw rows)
  ✓ Gross Profit via 'GrossProfit' (278 raw rows)
  ✓ Operating Income via 'OperatingIncomeLoss' (198 raw rows)
  ✓ Net Income via 'NetIncomeLoss' (281 raw rows)
  ✓ EPS Diluted via 'EarningsPerShareDiluted' (124 raw rows)

Amazon (AMZN)
  ✓ Revenue via 'SalesRevenueNet' (185 raw rows)
  ✓ Gross Profit via 'GrossProfit' (11 raw rows)
  ✓ Operating Income via 'OperatingIncomeLoss' (

---
## Section 4: Check 1 — Completeness

Validates that each company has the expected number of quarters and that no key fields are null.

In [5]:
print("CHECK 1: COMPLETENESS")
print("=" * 50)

EXPECTED_QUARTERS = 8  # 2 full years of quarterly data

for company, info in COMPANIES.items():
    ticker = info['ticker']
    raw = ALL_RAW.get(company, {})

    for metric, df in raw.items():
        # Filter to quarterly filings
        quarterly = df[df['form'].isin(['10-Q', '10-K'])].copy()
        if 'frame' in quarterly.columns:
            quarterly = quarterly[quarterly['frame'].str.contains('Q', na=False)]
        quarterly = quarterly.drop_duplicates(subset='end', keep='first')
        quarterly = quarterly.sort_values('end', ascending=False).head(EXPECTED_QUARTERS)

        actual_quarters = len(quarterly)

        # Check: enough quarters present?
        if actual_quarters < EXPECTED_QUARTERS:
            gap = EXPECTED_QUARTERS - actual_quarters
            severity = 'Critical' if gap >= 4 else ('High' if gap >= 2 else 'Medium')
            log_issue(company, ticker, metric, 'Multiple',
                      'Completeness', severity,
                      f'Only {actual_quarters} of {EXPECTED_QUARTERS} expected quarters found ({gap} missing)',
                      expected=EXPECTED_QUARTERS, actual=actual_quarters,
                      remediation='Verify filing history on EDGAR; company may have fewer filings or used different tags in earlier periods')
            print(f"  [{severity}] {company} / {metric}: {actual_quarters}/{EXPECTED_QUARTERS} quarters")

        # Check: null values in val column?
        null_count = quarterly['val'].isna().sum()
        if null_count > 0:
            log_issue(company, ticker, metric, 'Multiple',
                      'Completeness', 'High',
                      f'{null_count} null value(s) in reported data',
                      expected=0, actual=null_count,
                      remediation='Null values in XBRL data indicate unfiled or improperly tagged entries; cross-check 10-Q/10-K PDFs')
            print(f"  [High] {company} / {metric}: {null_count} null value(s)")

        # Check: filed date present?
        if 'filed' in quarterly.columns:
            missing_filed = quarterly['filed'].isna().sum()
            if missing_filed > 0:
                log_issue(company, ticker, metric, 'Multiple',
                          'Completeness', 'Low',
                          f'{missing_filed} row(s) missing filed date',
                          remediation='Filed date used for timeliness checks; retrieve from EDGAR submissions endpoint')

completeness_issues = [i for i in ISSUES if i['Check Category'] == 'Completeness']
print(f"\nCompleteness check complete. Issues found: {len(completeness_issues)}")

CHECK 1: COMPLETENESS
  [Critical] Amazon / Gross Profit: 4/8 quarters

Completeness check complete. Issues found: 1


---
## Section 5: Check 2 — Mathematical Consistency

Verifies that reported financial relationships hold. The fundamental check: **Revenue − Cost of Goods Sold = Gross Profit**. If this doesn't reconcile within tolerance, something is wrong with the data — either a wrong tag, a unit mismatch, or a restatement.

In [6]:
print("CHECK 2: MATHEMATICAL CONSISTENCY")
print("=" * 50)

def get_quarterly_series(company, metric, n=8):
    """Extract clean quarterly series for a given company and metric."""
    raw = ALL_RAW.get(company, {})
    if metric not in raw:
        return pd.Series(dtype=float)
    df = raw[metric].copy()
    quarterly = df[df['form'].isin(['10-Q', '10-K'])]
    if 'frame' in quarterly.columns:
        quarterly = quarterly[quarterly['frame'].str.contains('Q', na=False)]
    quarterly = quarterly.drop_duplicates(subset='end', keep='first')
    quarterly = quarterly.sort_values('end', ascending=False).head(n)
    return quarterly.set_index('end')['val'] / 1_000_000  # Convert to millions


for company, info in COMPANIES.items():
    ticker = info['ticker']

    rev = get_quarterly_series(company, 'Revenue')
    gp  = get_quarterly_series(company, 'Gross Profit')
    oi  = get_quarterly_series(company, 'Operating Income')
    ni  = get_quarterly_series(company, 'Net Income')

    # Check 2a: Gross Profit <= Revenue (always)
    if len(rev) > 0 and len(gp) > 0:
        common_idx = rev.index.intersection(gp.index)
        for period in common_idx:
            r, g = rev[period], gp[period]
            if pd.notna(r) and pd.notna(g) and g > r:
                log_issue(company, ticker, 'Gross Profit', period,
                          'Mathematical Consistency', 'Critical',
                          f'Gross Profit (${g:,.0f}M) exceeds Revenue (${r:,.0f}M) — impossible relationship',
                          expected=f'GP <= Rev (${r:,.0f}M)', actual=f'${g:,.0f}M',
                          remediation='Likely wrong XBRL tag; check whether company reports COGS separately and recalculate')
                print(f"  [Critical] {company} / {period.date()}: GP > Revenue")

    # Check 2b: Gross Margin bounds (0% to 100%)
    if len(rev) > 0 and len(gp) > 0:
        common_idx = rev.index.intersection(gp.index)
        for period in common_idx:
            r, g = rev[period], gp[period]
            if pd.notna(r) and pd.notna(g) and r != 0:
                margin = g / r
                if margin < 0 or margin > 1:
                    log_issue(company, ticker, 'Gross Margin', period,
                              'Mathematical Consistency', 'High',
                              f'Gross margin of {margin:.1%} is outside valid range (0%–100%)',
                              expected='0% – 100%', actual=f'{margin:.1%}',
                              remediation='Review whether gross profit or revenue was reported incorrectly; check for restatements')
                    print(f"  [High] {company} / {period.date()}: GM={margin:.1%}")

    # Check 2c: Operating Income <= Gross Profit (usually; exceptions exist)
    if len(gp) > 0 and len(oi) > 0:
        common_idx = gp.index.intersection(oi.index)
        for period in common_idx:
            g, o = gp[period], oi[period]
            if pd.notna(g) and pd.notna(o) and g > 0 and o > g * 1.10:  # >10% over GP is suspicious
                log_issue(company, ticker, 'Operating Income', period,
                          'Mathematical Consistency', 'Medium',
                          f'Operating Income (${o:,.0f}M) exceeds Gross Profit (${g:,.0f}M) by more than 10%',
                          expected=f'<= ${g:,.0f}M', actual=f'${o:,.0f}M',
                          remediation='Unusual but possible if "Other Income" items are large; verify against 10-Q income statement')
                print(f"  [Medium] {company} / {period.date()}: OI > GP")

math_issues = [i for i in ISSUES if i['Check Category'] == 'Mathematical Consistency']
print(f"\nMath consistency check complete. Issues found: {len(math_issues)}")

CHECK 2: MATHEMATICAL CONSISTENCY

Math consistency check complete. Issues found: 0


---
## Section 6: Check 3 — Cross-Filing Reconciliation

The most common source of silent errors in EDGAR data: **cumulative YTD values mistaken for quarterly values**. EDGAR returns both quarterly (`10-Q`) and annual (`10-K`) figures, and some rows represent cumulative year-to-date amounts rather than single-quarter amounts. This check verifies that quarterly figures, when summed, reasonably approximate the annual 10-K figure.

In [7]:
print("CHECK 3: CROSS-FILING RECONCILIATION")
print("=" * 50)
print("Verifying: Sum of quarterly figures ≈ Annual 10-K reported figure")
print()

def get_annual_series(company, metric):
    """Get annual 10-K values for reconciliation."""
    raw = ALL_RAW.get(company, {})
    if metric not in raw:
        return pd.Series(dtype=float)
    df = raw[metric].copy()
    annual = df[df['form'] == '10-K'].copy()
    annual = annual.drop_duplicates(subset='end', keep='first')
    annual = annual.sort_values('end', ascending=False)
    return annual.set_index('end')['val'] / 1_000_000


def get_quarterly_for_year(company, metric, year_end):
    """Sum quarterly values for the four quarters ending at year_end."""
    raw = ALL_RAW.get(company, {})
    if metric not in raw:
        return None
    df = raw[metric].copy()
    quarterly = df[df['form'] == '10-Q'].copy()
    if 'frame' in quarterly.columns:
        quarterly = quarterly[quarterly['frame'].str.contains('Q', na=False)]
    quarterly = quarterly.drop_duplicates(subset='end', keep='first')
    # Find four quarters in the 12 months before year_end
    mask = (quarterly['end'] <= year_end) & \
           (quarterly['end'] > year_end - pd.DateOffset(months=13))
    relevant = quarterly[mask]
    if len(relevant) < 3:  # Need at least 3 quarters to attempt reconciliation
        return None
    return relevant['val'].sum() / 1_000_000


tol = THRESHOLDS['recon_tolerance']

for company, info in COMPANIES.items():
    ticker = info['ticker']
    for metric in ['Revenue', 'Net Income', 'Operating Income']:
        annual = get_annual_series(company, metric)
        for year_end, annual_val in annual.head(2).items():  # Last 2 fiscal years
            quarterly_sum = get_quarterly_for_year(company, metric, year_end)
            if quarterly_sum is None or pd.isna(annual_val):
                continue
            diff_pct = abs(quarterly_sum - annual_val) / abs(annual_val) if annual_val != 0 else 0
            if diff_pct > tol:
                severity = 'Critical' if diff_pct > 0.20 else ('High' if diff_pct > 0.10 else 'Medium')
                log_issue(company, ticker, metric, year_end.date(),
                          'Cross-Filing Reconciliation', severity,
                          f'Quarterly sum (${quarterly_sum:,.0f}M) differs from 10-K annual (${annual_val:,.0f}M) by {diff_pct:.1%}',
                          expected=f'Within {tol:.0%} of ${annual_val:,.0f}M',
                          actual=f'${quarterly_sum:,.0f}M (Δ={diff_pct:.1%})',
                          remediation='Check whether any quarterly rows are cumulative YTD rather than single-quarter; filter by frame tag (CY20XXQ1 not CY20XXQ1I)')
                print(f"  [{severity}] {company} / {metric} / {year_end.date()}: Δ={diff_pct:.1%}")

recon_issues = [i for i in ISSUES if i['Check Category'] == 'Cross-Filing Reconciliation']
print(f"\nReconciliation check complete. Issues found: {len(recon_issues)}")

CHECK 3: CROSS-FILING RECONCILIATION
Verifying: Sum of quarterly figures ≈ Annual 10-K reported figure

  [Critical] Apple / Net Income / 2025-09-27: Δ=24.5%
  [High] Apple / Net Income / 2024-09-28: Δ=15.7%
  [Critical] Apple / Operating Income / 2025-09-27: Δ=24.4%
  [Critical] Apple / Operating Income / 2024-09-28: Δ=24.0%
  [Critical] Microsoft / Net Income / 2025-06-30: Δ=26.7%
  [Critical] Microsoft / Net Income / 2024-06-30: Δ=25.0%
  [Critical] Microsoft / Operating Income / 2025-06-30: Δ=26.7%
  [Critical] Microsoft / Operating Income / 2024-06-30: Δ=25.5%
  [Critical] Tesla / Revenue / 2025-12-31: Δ=26.3%
  [Critical] Tesla / Revenue / 2024-12-31: Δ=26.3%
  [Critical] Tesla / Net Income / 2025-12-31: Δ=22.1%
  [Critical] Tesla / Net Income / 2024-12-31: Δ=30.0%
  [Critical] Tesla / Operating Income / 2025-12-31: Δ=32.4%
  [Critical] Tesla / Operating Income / 2024-12-31: Δ=22.4%
  [Critical] Amazon / Net Income / 2025-12-31: Δ=27.3%
  [Critical] Amazon / Net Income / 2024-12-

---
## Section 7: Check 4 — Temporal Continuity

In [8]:
print("CHECK 4: TEMPORAL CONTINUITY")
print("=" * 50)

for company, info in COMPANIES.items():
    ticker = info['ticker']
    for metric, df_raw in ALL_RAW.get(company, {}).items():
        df = df_raw[df_raw['form'].isin(['10-Q', '10-K'])].copy()
        if 'frame' in df.columns:
            df = df[df['frame'].str.contains('Q', na=False)]
        df = df.drop_duplicates(subset='end', keep='first').sort_values('end')
        if len(df) < 2:
            continue
        periods = df['end'].tolist()
        for i in range(1, len(periods)):
            gap_days = (periods[i] - periods[i-1]).days
            # Expect ~90 days between quarters; flag if >130 (skipped quarter)
            if gap_days > 130:
                log_issue(company, ticker, metric,
                          f'{periods[i-1].date()} → {periods[i].date()}',
                          'Temporal Continuity', 'High',
                          f'Gap of {gap_days} days between consecutive periods (expected ~90)',
                          expected='~90 days', actual=f'{gap_days} days',
                          remediation='Possible missed quarter filing or fiscal year change; verify on EDGAR submissions endpoint')
                print(f"  [High] {company}/{metric}: {gap_days}-day gap at {periods[i].date()}")

        # Check: any period reported AFTER the filing date (data integrity issue)
        if 'filed' in df.columns:
            df['filed_dt'] = pd.to_datetime(df['filed'], errors='coerce')
            bad = df[df['end'] > df['filed_dt']]
            for _, row in bad.iterrows():
                log_issue(company, ticker, metric, row['end'].date(),
                          'Temporal Continuity', 'Medium',
                          f'Period end date ({row["end"].date()}) is after filed date ({row["filed_dt"].date() if pd.notna(row["filed_dt"]) else "unknown"})',
                          remediation='Possible data entry error in EDGAR; check submission metadata')

temporal_issues = [i for i in ISSUES if i['Check Category'] == 'Temporal Continuity']
print(f"\nTemporal continuity check complete. Issues found: {len(temporal_issues)}")

CHECK 4: TEMPORAL CONTINUITY
  [High] Apple/Gross Profit: 182-day gap at 2008-12-27
  [High] Apple/Gross Profit: 462-day gap at 2011-12-31
  [High] Apple/Gross Profit: 455-day gap at 2013-12-28
  [High] Apple/Gross Profit: 182-day gap at 2021-12-25
  [High] Apple/Gross Profit: 189-day gap at 2022-12-31
  [High] Apple/Gross Profit: 182-day gap at 2023-12-30
  [High] Apple/Gross Profit: 182-day gap at 2024-12-28
  [High] Apple/Gross Profit: 182-day gap at 2025-12-27
  [High] Apple/Operating Income: 182-day gap at 2008-12-27
  [High] Apple/Operating Income: 182-day gap at 2009-12-26
  [High] Apple/Operating Income: 182-day gap at 2010-12-25
  [High] Apple/Operating Income: 189-day gap at 2011-12-31
  [High] Apple/Operating Income: 182-day gap at 2012-12-29
  [High] Apple/Operating Income: 182-day gap at 2013-12-28
  [High] Apple/Operating Income: 182-day gap at 2014-12-27
  [High] Apple/Operating Income: 182-day gap at 2015-12-26
  [High] Apple/Operating Income: 189-day gap at 2016-12-31


---
## Section 8: Check 5 — Anomaly Detection

In [9]:
print("CHECK 5: ANOMALY DETECTION")
print("=" * 50)

qoq_thresh  = THRESHOLDS['qoq_swing_pct']
sigma_thresh = THRESHOLDS['sigma_threshold']

for company, info in COMPANIES.items():
    ticker = info['ticker']
    for metric in ['Revenue', 'Net Income', 'Gross Profit']:
        series = get_quarterly_series(company, metric, n=12)
        if len(series) < 4:
            continue
        series = series.sort_index()
        vals = series.dropna()

        # 5a: Statistical outliers (beyond 3σ)
        mu, sigma = vals.mean(), vals.std()
        if sigma > 0:
            for period, val in vals.items():
                z = abs(val - mu) / sigma
                if z > sigma_thresh:
                    log_issue(company, ticker, metric, period.date(),
                              'Anomaly Detection', 'High',
                              f'Value ${val:,.0f}M is {z:.1f}σ from mean (${mu:,.0f}M ± ${sigma:,.0f}M)',
                              expected=f'Within 3σ of ${mu:,.0f}M', actual=f'${val:,.0f}M (z={z:.1f})',
                              remediation='Investigate: could be one-time item (litigation, acquisition), restatement, or data error. Cross-check 10-Q footnotes.')
                    print(f"  [High] {company}/{metric}/{period.date()}: z={z:.1f}")

        # 5b: Large QoQ swings (>50%)
        qoq = vals.pct_change().dropna()
        for period, chg in qoq.items():
            if abs(chg) > qoq_thresh:
                severity = 'High' if abs(chg) > 1.0 else 'Medium'
                prev_val = vals.get(vals.index[vals.index < period][-1]) if any(vals.index < period) else None
                log_issue(company, ticker, metric, period.date(),
                          'Anomaly Detection', severity,
                          f'Quarter-over-quarter change of {chg:.1%} exceeds {qoq_thresh:.0%} threshold',
                          expected=f'|QoQ| < {qoq_thresh:.0%}',
                          actual=f'{chg:+.1%} (from ${prev_val:,.0f}M to ${vals[period]:,.0f}M)' if prev_val else f'{chg:+.1%}',
                          remediation='Verify whether change is legitimate (seasonal pattern, M&A, restructuring) or indicates corrupted data')
                print(f"  [{severity}] {company}/{metric}/{period.date()}: QoQ={chg:+.1%}")

        # 5c: Sign reversals (positive → negative revenue, for example)
        if metric == 'Revenue':
            negatives = vals[vals < 0]
            for period, val in negatives.items():
                log_issue(company, ticker, metric, period.date(),
                          'Anomaly Detection', 'Critical',
                          f'Negative revenue value (${val:,.0f}M) — revenue cannot be negative',
                          expected='> 0', actual=f'${val:,.0f}M',
                          remediation='Almost certainly a data error — wrong sign applied or wrong tag used. Replace with correct value from 10-Q.')
                print(f"  [Critical] {company}/Revenue/{period.date()}: Negative value")

anomaly_issues = [i for i in ISSUES if i['Check Category'] == 'Anomaly Detection']
print(f"\nAnomaly detection complete. Issues found: {len(anomaly_issues)}")

CHECK 5: ANOMALY DETECTION
  [Medium] Apple/Revenue/2017-12-30: QoQ=+67.9%
  [Medium] Apple/Net Income/2022-12-31: QoQ=+54.3%
  [Medium] Apple/Net Income/2023-12-30: QoQ=+70.6%
  [Medium] Apple/Net Income/2024-12-28: QoQ=+69.4%
  [Medium] Apple/Net Income/2025-12-27: QoQ=+79.6%
  [Medium] Apple/Gross Profit/2023-12-30: QoQ=+50.6%
  [Medium] Apple/Gross Profit/2025-12-27: QoQ=+58.4%
  [Medium] Tesla/Net Income/2024-09-30: QoQ=+55.2%
  [Medium] Tesla/Net Income/2025-03-31: QoQ=-81.2%
  [High] Tesla/Net Income/2025-06-30: QoQ=+186.6%
  [Medium] Tesla/Net Income/2026-03-31: QoQ=-65.3%
  [High] Amazon/Net Income/2022-09-30: QoQ=-241.6%
  [High] Amazon/Net Income/2023-06-30: QoQ=+112.8%
  [High] Johnson & Johnson/Net Income/2023-04-02: QoQ=-101.9%
  [High] Johnson & Johnson/Net Income/2023-07-02: QoQ=-7664.7%
  [High] Johnson & Johnson/Net Income/2023-10-01: QoQ=+406.0%
  [Medium] Johnson & Johnson/Net Income/2023-12-31: QoQ=-84.4%
  [High] Johnson & Johnson/Net Income/2025-03-30: QoQ=+308.3

---
## Section 9: Check 6 — XBRL Tag Consistency

In [10]:
print("CHECK 6: XBRL TAG CONSISTENCY")
print("=" * 50)
print("Checking whether the same metric is reported under different tags across years")
print()

for company, info in COMPANIES.items():
    ticker = info['ticker']
    for metric, tags in METRICS.items():
        found_tags = []
        for tag in tags:
            df = fetch_concept_raw(info['cik'], tag)
            time.sleep(0.1)
            if df is not None and len(df) > 0:
                quarterly = df[df['form'].isin(['10-Q', '10-K'])]
                if len(quarterly) > 0:
                    found_tags.append((tag, quarterly['end'].min(), quarterly['end'].max(), len(quarterly)))

        if len(found_tags) > 1:
            # Multiple tags found — check for temporal overlap
            tag_desc = ' | '.join([f"{t[0]} ({t[3]} rows, {t[1].year}–{t[2].year})" for t in found_tags])
            log_issue(company, ticker, metric, 'Multiple periods',
                      'XBRL Tag Consistency', 'Medium',
                      f'Metric reported under {len(found_tags)} different XBRL tags across filing history: {tag_desc}',
                      expected='Single consistent tag', actual=f'{len(found_tags)} tags',
                      remediation='Company changed XBRL taxonomy. Use most recent tag for current data; splice historical series carefully, validating overlap years against 10-K PDFs.')
            print(f"  [Medium] {company}/{metric}: {len(found_tags)} tags found")
            for tag_info in found_tags:
                print(f"    → {tag_info[0]}: {tag_info[3]} rows, {tag_info[1].year}–{tag_info[2].year}")

tag_issues = [i for i in ISSUES if i['Check Category'] == 'XBRL Tag Consistency']
print(f"\nXBRL tag check complete. Issues found: {len(tag_issues)}")

CHECK 6: XBRL TAG CONSISTENCY
Checking whether the same metric is reported under different tags across years

  [Medium] Apple/Revenue: 3 tags found
    → Revenues: 11 rows, 2016–2018
    → SalesRevenueNet: 185 rows, 2007–2018
    → RevenueFromContractWithCustomerExcludingAssessedTax: 113 rows, 2017–2026
  [Medium] Microsoft/Revenue: 3 tags found
    → Revenues: 31 rows, 2007–2010
    → SalesRevenueNet: 147 rows, 2009–2018
    → RevenueFromContractWithCustomerExcludingAssessedTax: 128 rows, 2016–2026
  [Medium] Tesla/Revenue: 2 tags found
    → Revenues: 267 rows, 2009–2026
    → RevenueFromContractWithCustomerExcludingAssessedTax: 41 rows, 2018–2026
  [Medium] Amazon/Revenue: 2 tags found
    → SalesRevenueNet: 185 rows, 2007–2018
    → RevenueFromContractWithCustomerExcludingAssessedTax: 124 rows, 2016–2026
  [Medium] Johnson & Johnson/Revenue: 2 tags found
    → Revenues: 18 rows, 2012–2014
    → RevenueFromContractWithCustomerExcludingAssessedTax: 124 rows, 2017–2026

XBRL tag chec

---
## Section 10: Check 7 — Duplicate Detection

In [11]:
print("CHECK 7: DUPLICATE DETECTION")
print("=" * 50)

for company, info in COMPANIES.items():
    ticker = info['ticker']
    for metric, df_raw in ALL_RAW.get(company, {}).items():
        df = df_raw[df_raw['form'].isin(['10-Q', '10-K'])].copy()
        if 'frame' in df.columns:
            df = df[df['frame'].str.contains('Q', na=False)]

        # Find periods with more than one entry
        dupes = df[df.duplicated(subset='end', keep=False)]
        if len(dupes) == 0:
            continue

        dupe_periods = dupes['end'].unique()
        for period in dupe_periods:
            period_rows = dupes[dupes['end'] == period]
            vals = period_rows['val'].tolist()
            all_same = len(set(vals)) == 1

            severity = 'Low' if all_same else 'Critical'
            desc = (f'{len(period_rows)} entries for period {pd.Timestamp(period).date()} with '
                    f'{"identical" if all_same else "CONFLICTING"} values: {[f"${v/1e6:,.0f}M" for v in vals]}')
            remediation = ('Duplicate with identical values — safe to deduplicate by keeping most recently filed entry'
                           if all_same else
                           'CONFLICTING duplicate values — cannot auto-resolve; manually verify correct value against 10-Q PDF before use')

            log_issue(company, ticker, metric, pd.Timestamp(period).date(),
                      'Duplicate Detection', severity, desc,
                      expected='1 entry per period', actual=f'{len(period_rows)} entries',
                      remediation=remediation)
            print(f"  [{severity}] {company}/{metric}/{pd.Timestamp(period).date()}: {len(period_rows)} entries, {'same' if all_same else 'CONFLICTING'} values")

dupe_issues = [i for i in ISSUES if i['Check Category'] == 'Duplicate Detection']
print(f"\nDuplicate check complete. Issues found: {len(dupe_issues)}")

CHECK 7: DUPLICATE DETECTION

Duplicate check complete. Issues found: 0


---
## Section 11: Audit Summary & Scoring

In [12]:
issues_df = pd.DataFrame(ISSUES)

print("AUDIT SUMMARY")
print("=" * 60)
print(f"Total issues logged: {len(issues_df)}")
print()

if len(issues_df) > 0:
    print("By Severity:")
    print(issues_df['Severity'].value_counts().to_string())
    print()
    print("By Check Category:")
    print(issues_df['Check Category'].value_counts().to_string())
    print()
    print("By Company:")
    print(issues_df['Company'].value_counts().to_string())


def compute_dq_score(company_issues, total_checks=35):
    """
    Data Quality Score (0–100).
    Starts at 100, deducts weighted points per issue.
    Critical=-10, High=-5, Medium=-2, Low=-1
    """
    penalty = sum(SEVERITY_WEIGHTS.get(i['Severity'], 0) for i in company_issues)
    return max(0, 100 - penalty)


SCORES = {}
print("\nData Quality Scores by Company:")
print("-" * 40)
for company in COMPANIES:
    co_issues = [i for i in ISSUES if i['Company'] == company]
    score = compute_dq_score(co_issues)
    SCORES[company] = score
    rating = '🟢 PASS' if score >= 80 else ('🟡 REVIEW' if score >= 60 else '🔴 FAIL')
    print(f"  {company:<25} {score:>3}/100  {rating}  ({len(co_issues)} issues)")

AUDIT SUMMARY
Total issues logged: 209

By Severity:
Severity
High        174
Critical     19
Medium       16

By Check Category:
Check Category
Temporal Continuity            163
Cross-Filing Reconciliation     22
Anomaly Detection               18
XBRL Tag Consistency             5
Completeness                     1

By Company:
Company
Tesla                56
Apple                54
Microsoft            37
Amazon               31
Johnson & Johnson    31

Data Quality Scores by Company:
----------------------------------------
  Apple                       0/100  🔴 FAIL  (54 issues)
  Microsoft                   0/100  🔴 FAIL  (37 issues)
  Tesla                       0/100  🔴 FAIL  (56 issues)
  Amazon                      0/100  🔴 FAIL  (31 issues)
  Johnson & Johnson           0/100  🔴 FAIL  (31 issues)


---
## Section 12: Export Excel Audit Report

In [13]:
# ── Style constants ────────────────────────────────────────────────────────────
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

FILL_NAVY    = PatternFill('solid', start_color='1F3864')
FILL_BLUE    = PatternFill('solid', start_color='2E75B6')
FILL_LTBLUE  = PatternFill('solid', start_color='D6E4F0')
FILL_ALT     = PatternFill('solid', start_color='F2F7FB')
FILL_CRIT    = PatternFill('solid', start_color='FCE4D6')  # Critical
FILL_HIGH    = PatternFill('solid', start_color='FFF2CC')  # High
FILL_MED     = PatternFill('solid', start_color='EBF3FB')  # Medium
FILL_LOW     = PatternFill('solid', start_color='E2EFDA')  # Low
FILL_PASS    = PatternFill('solid', start_color='E2EFDA')
FILL_REVIEW  = PatternFill('solid', start_color='FFF2CC')
FILL_FAIL    = PatternFill('solid', start_color='FCE4D6')

F_WHITE_BOLD = Font(name='Arial', bold=True, color='FFFFFF', size=10)
F_TITLE      = Font(name='Arial', bold=True, color='1F3864', size=14)
F_SUBHDR     = Font(name='Arial', bold=True, color='2E75B6', size=10)
F_BODY       = Font(name='Arial', size=10)
F_BOLD       = Font(name='Arial', bold=True, size=10)
F_GRAY       = Font(name='Arial', color='595959', size=8, italic=True)
F_RED        = Font(name='Arial', bold=True, color='C00000', size=10)
F_ORANGE     = Font(name='Arial', bold=True, color='E26B0A', size=10)
F_GREEN      = Font(name='Arial', bold=True, color='375623', size=10)

THIN   = Side(style='thin',   color='BFBFBF')
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)
CENTER = Alignment(horizontal='center', vertical='center', wrap_text=True)
LEFT   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
RIGHT  = Alignment(horizontal='right',  vertical='center')


def hdr(ws, row, col, val, fill=FILL_BLUE, h=22):
    ws.row_dimensions[row].height = h
    c = ws.cell(row=row, column=col, value=val)
    c.fill, c.font, c.alignment, c.border = fill, F_WHITE_BOLD, CENTER, BORDER
    return c

def dat(ws, row, col, val, font=F_BODY, align=LEFT, fmt=None, fill=None, h=None):
    if h: ws.row_dimensions[row].height = h
    c = ws.cell(row=row, column=col, value=val)
    c.font, c.alignment, c.border = font, align, BORDER
    if fmt:  c.number_format = fmt
    if fill: c.fill = fill
    return c

SEVERITY_FILL = {'Critical': FILL_CRIT, 'High': FILL_HIGH, 'Medium': FILL_MED, 'Low': FILL_LOW}
SEVERITY_FONT = {'Critical': F_RED, 'High': F_ORANGE, 'Medium': F_BODY, 'Low': F_GREEN}

print("Excel styles defined.")

Excel styles defined.


In [14]:
wb = Workbook()

# ════════════════════════════════════════════════════════
# SHEET 1: EXECUTIVE SUMMARY
# ════════════════════════════════════════════════════════
ws_exec = wb.active
ws_exec.title = 'Executive Summary'
ws_exec.sheet_view.showGridLines = False

for col, w in [('A',4),('B',26),('C',12),('D',10),('E',10),('F',10),('G',10),('H',10),('I',12)]:
    ws_exec.column_dimensions[col].width = w

ws_exec.row_dimensions[1].height = 8
ws_exec.row_dimensions[2].height = 36
ws_exec.merge_cells('B2:I2')
ws_exec['B2'].value = 'Financial Data Quality Audit Report — SEC EDGAR Multi-Company Review'
ws_exec['B2'].font = F_TITLE
ws_exec['B2'].alignment = LEFT

ws_exec.row_dimensions[3].height = 14
ws_exec.merge_cells('B3:I3')
ws_exec['B3'].value = f'Audit Date: {AUDIT_DATE}  |  Companies: AAPL · MSFT · TSLA · AMZN · JNJ  |  Data Source: SEC EDGAR XBRL API'
ws_exec['B3'].font = F_GRAY

ws_exec.row_dimensions[4].height = 14
ws_exec.merge_cells('B4:I4')
ws_exec['B4'].value = 'Scope: 7 audit check categories · 5 financial metrics · 8 quarters of history · Issue severity: Critical / High / Medium / Low'
ws_exec['B4'].font = F_GRAY

# Summary stats
ws_exec.row_dimensions[6].height = 22
ws_exec.merge_cells('B6:I6')
ws_exec['B6'].value = 'AUDIT SUMMARY'
ws_exec['B6'].fill = FILL_NAVY
ws_exec['B6'].font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
ws_exec['B6'].alignment = LEFT

stat_hdrs = ['Total Issues', 'Critical', 'High', 'Medium', 'Low', 'Companies Audited', 'Checks Run', 'Audit Date']
for i, h in enumerate(stat_hdrs, 2):
    hdr(ws_exec, 7, i, h)

ws_exec.row_dimensions[8].height = 22
if len(issues_df) > 0:
    sev_counts = issues_df['Severity'].value_counts()
    stat_vals = [
        len(issues_df),
        sev_counts.get('Critical', 0),
        sev_counts.get('High', 0),
        sev_counts.get('Medium', 0),
        sev_counts.get('Low', 0),
        len(COMPANIES), 7, AUDIT_DATE
    ]
else:
    stat_vals = [0, 0, 0, 0, 0, len(COMPANIES), 7, AUDIT_DATE]

for i, v in enumerate(stat_vals, 2):
    fill = None
    font = F_BOLD
    if i == 3 and v > 0: fill, font = FILL_CRIT, F_RED
    elif i == 4 and v > 0: fill, font = FILL_HIGH, F_ORANGE
    dat(ws_exec, 8, i, v, font=font, align=CENTER, fill=fill)

# Company scorecard
ws_exec.row_dimensions[10].height = 22
ws_exec.merge_cells('B10:I10')
ws_exec['B10'].value = 'DATA QUALITY SCORECARD BY COMPANY'
ws_exec['B10'].fill = FILL_NAVY
ws_exec['B10'].font = Font(name='Arial', bold=True, color='FFFFFF', size=11)
ws_exec['B10'].alignment = LEFT

score_hdrs = ['Company', 'Ticker', 'DQ Score', 'Rating', 'Critical', 'High', 'Medium', 'Low']
for i, h in enumerate(score_hdrs, 2):
    hdr(ws_exec, 11, i, h)

for r_idx, (company, info) in enumerate(COMPANIES.items()):
    co_issues = [i for i in ISSUES if i['Company'] == company]
    score = SCORES.get(company, 100)
    sev_co = pd.Series([i['Severity'] for i in co_issues]).value_counts() if co_issues else pd.Series()
    rating = 'PASS' if score >= 80 else ('REVIEW' if score >= 60 else 'FAIL')
    score_fill = FILL_PASS if score >= 80 else (FILL_REVIEW if score >= 60 else FILL_FAIL)
    excel_row = 12 + r_idx
    ws_exec.row_dimensions[excel_row].height = 18
    row_vals = [
        (company,                        F_BOLD,  LEFT),
        (info['ticker'],                 F_BODY,  CENTER),
        (f'{score}/100',                 F_BOLD,  CENTER),
        (rating,                         F_BOLD,  CENTER),
        (sev_co.get('Critical', 0),      F_RED if sev_co.get('Critical',0)>0 else F_BODY, CENTER),
        (sev_co.get('High', 0),          F_ORANGE if sev_co.get('High',0)>0 else F_BODY, CENTER),
        (sev_co.get('Medium', 0),        F_BODY,  CENTER),
        (sev_co.get('Low', 0),           F_GREEN if sev_co.get('Low',0)>0 else F_BODY, CENTER),
    ]
    for c_idx, (val, font, align) in enumerate(row_vals, 2):
        fill = score_fill if c_idx == 5 else (FILL_ALT if r_idx % 2 == 1 else None)
        dat(ws_exec, excel_row, c_idx, val, font=font, align=align, fill=fill)

# Score legend
legend_row = 12 + len(COMPANIES) + 2
ws_exec.merge_cells(f'B{legend_row}:I{legend_row}')
ws_exec[f'B{legend_row}'].value = 'Score Legend: 🟢 PASS = 80–100 (production-ready with monitoring)  |  🟡 REVIEW = 60–79 (use with documented caveats)  |  🔴 FAIL = <60 (remediate before use)'
ws_exec[f'B{legend_row}'].font = F_GRAY

print("Executive Summary sheet built.")

Executive Summary sheet built.


In [15]:
# ════════════════════════════════════════════════════════
# SHEET 2: ISSUE LOG
# ════════════════════════════════════════════════════════
ws_log = wb.create_sheet('Issue Log')
ws_log.sheet_view.showGridLines = False
ws_log.freeze_panes = 'A6'  # Freeze header row

log_col_widths = [10,18,8,20,14,22,10,45,20,20,50]
for i, w in enumerate(log_col_widths, 1):
    ws_log.column_dimensions[get_column_letter(i)].width = w

ws_log.row_dimensions[1].height = 8
ws_log.row_dimensions[2].height = 28
ws_log.merge_cells('A2:K2')
ws_log['A2'].value = 'Issue Log — All Data Quality Findings'
ws_log['A2'].font = F_TITLE
ws_log.row_dimensions[3].height = 14
ws_log.merge_cells('A3:K3')
ws_log['A3'].value = f'Sorted by Severity (Critical first) | Audit Date: {AUDIT_DATE} | Total issues: {len(ISSUES)}'
ws_log['A3'].font = F_GRAY
ws_log.row_dimensions[4].height = 14
ws_log.merge_cells('A4:K4')
ws_log['A4'].value = 'Color key: Red = Critical  |  Orange/Yellow = High  |  Blue = Medium  |  Green = Low'
ws_log['A4'].font = F_GRAY

log_headers = ['Audit Date','Company','Ticker','Metric','Period',
               'Check Category','Severity','Description','Expected Value','Actual Value','Remediation']
for i, h in enumerate(log_headers, 1):
    hdr(ws_log, 5, i, h, fill=FILL_NAVY)

# Sort by severity
sev_order = {'Critical': 0, 'High': 1, 'Medium': 2, 'Low': 3}
sorted_issues = sorted(ISSUES, key=lambda x: (sev_order.get(x['Severity'], 4), x['Company']))

for r_idx, issue in enumerate(sorted_issues):
    excel_row = 6 + r_idx
    ws_log.row_dimensions[excel_row].height = 32
    sev = issue['Severity']
    row_fill = SEVERITY_FILL.get(sev)
    for c_idx, key in enumerate(log_headers, 1):
        font = SEVERITY_FONT.get(sev, F_BODY) if c_idx == 7 else F_BODY
        dat(ws_log, excel_row, c_idx, issue.get(key, ''),
            font=font, align=LEFT if c_idx > 3 else CENTER,
            fill=row_fill if c_idx == 7 else (FILL_ALT if r_idx % 2 == 1 else None))

print("Issue Log sheet built.")

Issue Log sheet built.


In [16]:
# ════════════════════════════════════════════════════════
# SHEET 3: CHECK SUMMARY (pivot by check category)
# ════════════════════════════════════════════════════════
ws_chk = wb.create_sheet('Check Summary')
ws_chk.sheet_view.showGridLines = False

for col, w in [('A',28),('B',12),('C',12),('D',12),('E',12),('F',12),('G',16)]:
    ws_chk.column_dimensions[col].width = w

ws_chk.row_dimensions[1].height = 8
ws_chk.row_dimensions[2].height = 28
ws_chk.merge_cells('A2:G2')
ws_chk['A2'].value = 'Check Summary — Issues by Category'
ws_chk['A2'].font = F_TITLE

check_hdrs = ['Check Category', 'Critical', 'High', 'Medium', 'Low', 'Total', 'Status']
for i, h in enumerate(check_hdrs, 1):
    hdr(ws_chk, 4, i, h)

check_categories = [
    'Completeness', 'Mathematical Consistency', 'Cross-Filing Reconciliation',
    'Temporal Continuity', 'Anomaly Detection', 'XBRL Tag Consistency', 'Duplicate Detection'
]

for r_idx, cat in enumerate(check_categories):
    cat_issues = [i for i in ISSUES if i['Check Category'] == cat]
    sev_counts = pd.Series([i['Severity'] for i in cat_issues]).value_counts() if cat_issues else pd.Series()
    total = len(cat_issues)
    status = 'PASS ✓' if total == 0 else ('FAIL ✗' if sev_counts.get('Critical', 0) > 0 else 'REVIEW ⚠')
    status_fill = FILL_PASS if status.startswith('PASS') else (FILL_CRIT if status.startswith('FAIL') else FILL_REVIEW)
    excel_row = 5 + r_idx
    ws_chk.row_dimensions[excel_row].height = 20
    alt = r_idx % 2 == 1
    dat(ws_chk, excel_row, 1, cat, font=F_BOLD, align=LEFT, fill=FILL_ALT if alt else None)
    dat(ws_chk, excel_row, 2, sev_counts.get('Critical',0), font=F_RED if sev_counts.get('Critical',0)>0 else F_BODY, align=CENTER)
    dat(ws_chk, excel_row, 3, sev_counts.get('High',0),     font=F_ORANGE if sev_counts.get('High',0)>0 else F_BODY, align=CENTER)
    dat(ws_chk, excel_row, 4, sev_counts.get('Medium',0),   font=F_BODY, align=CENTER)
    dat(ws_chk, excel_row, 5, sev_counts.get('Low',0),      font=F_GREEN if sev_counts.get('Low',0)>0 else F_BODY, align=CENTER)
    dat(ws_chk, excel_row, 6, total, font=F_BOLD, align=CENTER)
    dat(ws_chk, excel_row, 7, status, font=F_BOLD, align=CENTER, fill=status_fill)

# Totals row
total_row = 5 + len(check_categories)
ws_chk.row_dimensions[total_row].height = 22
dat(ws_chk, total_row, 1, 'TOTAL', font=F_BOLD, align=LEFT, fill=FILL_LTBLUE)
for c_idx in range(2, 8):
    formula = f'=SUM({get_column_letter(c_idx)}5:{get_column_letter(c_idx)}{total_row-1})'
    dat(ws_chk, total_row, c_idx, formula, font=F_BOLD, align=CENTER, fill=FILL_LTBLUE)

print("Check Summary sheet built.")

Check Summary sheet built.


In [17]:
# ════════════════════════════════════════════════════════
# SHEET 4: REMEDIATION TRACKER
# ════════════════════════════════════════════════════════
ws_rem = wb.create_sheet('Remediation Tracker')
ws_rem.sheet_view.showGridLines = False

for col, w in [('A',5),('B',20),('C',8),('D',22),('E',10),('F',14),('G',45),('H',16),('I',16),('J',16)]:
    ws_rem.column_dimensions[col].width = w

ws_rem.row_dimensions[1].height = 8
ws_rem.row_dimensions[2].height = 28
ws_rem.merge_cells('B2:J2')
ws_rem['B2'].value = 'Remediation Tracker — Critical & High Issues'
ws_rem['B2'].font = F_TITLE
ws_rem.row_dimensions[3].height = 14
ws_rem.merge_cells('B3:J3')
ws_rem['B3'].value = 'Status column: Open → In Progress → Resolved | Update as issues are addressed'
ws_rem['B3'].font = F_GRAY

rem_hdrs = ['#','Company','Ticker','Check Category','Severity','Period','Remediation Action','Status','Owner','Due Date']
for i, h in enumerate(rem_hdrs, 2):
    hdr(ws_rem, 5, i, h, fill=FILL_NAVY)

priority_issues = [i for i in sorted_issues if i['Severity'] in ('Critical', 'High')]
for r_idx, issue in enumerate(priority_issues):
    excel_row = 6 + r_idx
    ws_rem.row_dimensions[excel_row].height = 34
    sev = issue['Severity']
    alt = r_idx % 2 == 1
    vals = [
        (r_idx+1,               F_BODY,  CENTER),
        (issue['Company'],      F_BOLD,  LEFT),
        (issue['Ticker'],       F_BODY,  CENTER),
        (issue['Check Category'], F_BODY, LEFT),
        (issue['Severity'],     SEVERITY_FONT[sev], CENTER),
        (issue['Period'],       F_BODY,  CENTER),
        (issue['Remediation'],  F_BODY,  LEFT),
        ('Open',                F_BODY,  CENTER),  # Status — to be updated
        ('',                    F_BODY,  CENTER),  # Owner
        ('',                    F_BODY,  CENTER),  # Due Date
    ]
    for c_idx, (val, font, align) in enumerate(vals, 2):
        fill = SEVERITY_FILL[sev] if c_idx == 6 else (FILL_ALT if alt else None)
        dat(ws_rem, excel_row, c_idx, val, font=font, align=align, fill=fill)

print(f"Remediation Tracker sheet built. {len(priority_issues)} Critical/High issues tracked.")

Remediation Tracker sheet built. 193 Critical/High issues tracked.


In [18]:
# ── Save ──────────────────────────────────────────────────────────────────────
output_path = 'Financial_Data_Quality_Audit.xlsx'
wb.save(output_path)

print(f"{'='*55}")
print(f"✓ Audit report saved: {output_path}")
print(f"  Sheets: Executive Summary | Issue Log | Check Summary | Remediation Tracker")
print(f"  Total issues logged: {len(ISSUES)}")
print(f"  Critical: {sum(1 for i in ISSUES if i['Severity']=='Critical')}  "
      f"High: {sum(1 for i in ISSUES if i['Severity']=='High')}  "
      f"Medium: {sum(1 for i in ISSUES if i['Severity']=='Medium')}  "
      f"Low: {sum(1 for i in ISSUES if i['Severity']=='Low')}")
print(f"{'='*55}")

✓ Audit report saved: Financial_Data_Quality_Audit.xlsx
  Sheets: Executive Summary | Issue Log | Check Summary | Remediation Tracker
  Total issues logged: 209
  Critical: 19  High: 174  Medium: 16  Low: 0


---
## Section 13: Key Findings & Narrative

*(Complete after running the notebook — findings will reflect live EDGAR data)*

### Why Data Quality Auditing Matters Before Financial Analysis

Every financial model is only as reliable as its inputs. In the SEC EDGAR context, three categories of issues occur frequently enough to warrant systematic checking on every data pull:

**1. Cumulative YTD vs. Quarterly values** — EDGAR's XBRL API returns both. A Q3 YTD figure of $90B revenue looks like a strong quarter; the actual Q3 standalone figure might be $28B. This is the most common source of inflated revenue figures in EDGAR-pulled data.

**2. XBRL tag changes** — Companies occasionally update their taxonomy mapping (often during system upgrades or following SEC guidance). Apple switched from `SalesRevenueNet` to `RevenueFromContractWithCustomerExcludingAssessedTax` in 2019. An analyst pulling `Revenues` only gets data through a certain year; the remainder of the history is blank — a completeness issue that's easy to miss.

**3. Refilings** — Companies refile quarters when correcting errors. EDGAR keeps all versions. Pulling the latest filed value (correct approach) vs. the first filed value can return meaningfully different numbers, particularly for companies that have restated earnings.

### Audit-First Workflow

The correct sequence for any financial data project:

```
1. Ingest raw data           ← Minimal filtering; preserve everything
2. Run audit checks          ← This notebook
3. Document all issues       ← Issue Log + Remediation Tracker
4. Remediate Critical/High   ← Before any analysis touches the data
5. Analyze clean data        ← SEC Financial Dashboard project
6. Note remaining caveats    ← In analysis narrative
```

Steps 1–4 are invisible in most portfolio projects. This notebook makes them visible — and that's the point.